# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [36]:
!pip -q install duckdb

In [37]:
import duckdb
import pandas as pd
import numpy as np
import os

con = duckdb.connect()

print("DuckDB ready")

DuckDB ready


In [38]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))

HF_TOKEN available: True


In [39]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face access configured.")

Hugging Face access configured.


## 1. My rule and its reason codes

### Signal check 1 — Staleness

**Verdict: MIXED**

Staleness is useful but not sufficient by itself. Pages updated 91–180 days ago still show meaningful visibility, with 62.5% receiving at least 100 impressions. However, this falls sharply for older pages: only 4.5% of pages in the 181–365 day bucket have at least 100 impressions, and none of the 365+ day pages do.

This suggests that old content is not automatically a refresh opportunity. Staleness is more useful when combined with a visibility signal.

### Signal check 2 — Impressions

**Verdict: CONFIRMED**

Impressions provide a useful visibility signal for prioritizing refresh work. Pages with substantial impression volume still have room for ranking improvement: 42.9% of pages with 3,000–29,999 impressions rank worse than position 10, while 22.5% of pages with 30,000+ impressions rank worse than position 10.

Therefore, impressions are a reasonable signal for finding content that has search visibility and may benefit from improvement.

### Baseline rule

A content page should be prioritized for refresh when it has meaningful search visibility, is not ranking strongly, and has not been updated recently.

The score combines:
- search impressions,
- average search position,
- days since last update.

The rule is transparent and uses only information available at the scoring snapshot.

**Reason Code:** `REFRESH_STALE_HIGH_IMPRESSIONS`

**Action:** `REFRESH_CONTENT`

### Build baseline_df

In [40]:

baseline_df = con.execute("""
WITH performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ), 0
        ) AS avg_position_90d

    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'
    ])

    WHERE gsc_data_available = TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        content_type,
        search_volume,
        competition,
        competition_level,
        main_intent,
        word_count,
        char_count,
        is_published,
        is_deleted

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
)

SELECT
    p.*,
    c.content_updated_date,
    c.content_type,
    c.search_volume,
    c.competition,
    c.competition_level,
    c.main_intent,
    c.word_count,
    c.char_count,
    c.is_published,
    c.is_deleted,

    DATE_DIFF(
        'day',
        CAST(c.content_updated_date AS DATE),
        DATE '2026-06-30'
    ) AS days_since_last_update

FROM performance p

INNER JOIN content c
    ON p.client_hash_id = c.client_hash_id
    AND p.content_hash_id = c.content_hash_id

WHERE
    c.content_updated_date IS NOT NULL
    AND CAST(c.content_updated_date AS DATE) <= DATE '2026-06-30'
    AND c.is_deleted = FALSE
    AND c.is_published = TRUE
""").df()

print("Baseline rows:", len(baseline_df))
print("Columns:", baseline_df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline rows: 215505
Columns: ['client_hash_id', 'content_hash_id', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'main_intent', 'word_count', 'char_count', 'is_published', 'is_deleted', 'days_since_last_update']


In [41]:
print(baseline_df[[
    "impressions_90d",
    "avg_position_90d",
    "days_since_last_update"
]].describe())

       impressions_90d  avg_position_90d  days_since_last_update
count     2.155050e+05     198593.000000           215505.000000
mean      2.786538e+03         20.440267               47.870133
std       1.308067e+04         19.035501               37.569809
min       1.000000e+00          0.128788                0.000000
25%       1.400000e+01          7.376942               29.000000
50%       1.810000e+02         12.750000               41.000000
75%       1.290000e+03         27.328021               41.000000
max       1.986586e+06        303.500000              394.000000


### Staleness Bucket

In [47]:
baseline_df["staleness_bucket"] = pd.cut(
    baseline_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
)

staleness_check = (
    baseline_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

print("Verdict: MIXED")
display(staleness_check)

Verdict: MIXED


,staleness_bucket,n,median_impressions
0,0-30 days,57560,849.5
1,31-90 days,130101,69.0
2,91-180 days,25702,543.0
3,181-365 days,2101,3.0
4,365+ days,41,4.0


### Impression bucket

In [48]:
baseline_df["impression_bucket"] = pd.cut(
    baseline_df["impressions_90d"],
    bins=[-1, 99, 299, 2999, 29999, float("inf")],
    labels=["<100", "100-299", "300-2,999", "3,000-29,999", "30,000+"]
)

impression_check = (
    baseline_df
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("avg_position_90d", "median")
    )
    .reset_index()
)

print("Verdict: CONFIRMED")
display(impression_check)

Verdict: CONFIRMED


,impression_bucket,n,median_position
0,<100,93080,11.479130
1,100-299,28076,19.500000
2,"300-2,999",61027,14.937318
3,"3,000-29,999",29755,8.864637
4,"30,000+",3567,6.679244


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [49]:
import os
import numpy as np
from sklearn.preprocessing import MinMaxScaler


ranked = baseline_df.copy()

ranked = ranked[ranked["avg_position_90d"] > 0].copy()

scaler = MinMaxScaler()

ranked[["impressions_score",
        "position_score",
        "staleness_score"]] = scaler.fit_transform(
    np.column_stack([
        np.log1p(ranked["impressions_90d"]),
        ranked["avg_position_90d"],
        ranked["days_since_last_update"]
    ])
)

# Transparent baseline score
ranked["baseline_score"] = (
      0.50 * ranked["impressions_score"]
    + 0.30 * ranked["position_score"]
    + 0.20 * ranked["staleness_score"]
)


ranked["reason_code"] = "REFRESH_STALE_HIGH_IMPRESSIONS"
ranked["action"] = "REFRESH_CONTENT"

ranked = ranked.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Write ranked queue
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
print("Ranked rows:", len(ranked))

print(
    ranked[
        ["rank",
         "content_hash_id",
         "baseline_score",
         "reason_code",
         "action"]
    ].head(10)
)

CSV saved successfully.
Ranked rows: 198593
   rank           content_hash_id  baseline_score  \
0     1  content_f43118e089ecc69a        0.520479   
1     2  content_c60628276389acbb        0.517759   
2     3  content_eadb33b5df496f4a        0.511255   
3     4  content_65b8a4998e633d89        0.510689   
4     5  content_a9322b74ca7cb1bb        0.508040   
5     6  content_e9f2d0579387d3c3        0.503470   
6     7  content_f2388a4b87a3b1dc        0.503286   
7     8  content_99fc6465edb0e52c        0.503033   
8     9  content_ac7b77e81c53d636        0.502343   
9    10  content_f33ad8a343180e8b        0.501966   

                      reason_code           action  
0  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
1  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
2  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
3  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
4  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
5  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT  
6

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [50]:
top10 = ranked.head(10).copy()

for i, row in top10.iterrows():

    print("=" * 80)
    print(f"Rank {i + 1}")
    print(f"Content: {row['content_hash_id']}")
    print(f"Action: {row['action']}")
    print(f"Reason: {row['reason_code']}")

    print(
        f"Why it's here: It has a high baseline score "
        f"({row['baseline_score']:.3f}) based on search impressions, "
        f"average position, and content staleness."
    )

    print(
        "What would make it wrong: The page may already be accurate and "
        "up-to-date, its traffic may be seasonal, or its ranking may be "
        "limited by competition rather than content quality."
    )

    print()

Rank 1
Content: content_f43118e089ecc69a
Action: REFRESH_CONTENT
Reason: REFRESH_STALE_HIGH_IMPRESSIONS
Why it's here: It has a high baseline score (0.520) based on search impressions, average position, and content staleness.
What would make it wrong: The page may already be accurate and up-to-date, its traffic may be seasonal, or its ranking may be limited by competition rather than content quality.

Rank 2
Content: content_c60628276389acbb
Action: REFRESH_CONTENT
Reason: REFRESH_STALE_HIGH_IMPRESSIONS
Why it's here: It has a high baseline score (0.518) based on search impressions, average position, and content staleness.
What would make it wrong: The page may already be accurate and up-to-date, its traffic may be seasonal, or its ranking may be limited by competition rather than content quality.

Rank 3
Content: content_eadb33b5df496f4a
Action: REFRESH_CONTENT
Reason: REFRESH_STALE_HIGH_IMPRESSIONS
Why it's here: It has a high baseline score (0.511) based on search impressions, avera

## 4. Weak picks + leakage check

### Weak Picks

The baseline rule is simple and transparent, but it can produce weak picks.

Possible weak picks include:

- Pages with high impressions that are already accurate and up to date.
- Seasonal pages where traffic patterns are temporary.
- Pages with poor rankings because of strong competition rather than outdated content.
- Pages with high impressions but low business value.

The rule can also miss useful refresh opportunities when a page has low impressions but would benefit from an update.

### Leakage Check

The baseline uses only information available at the time of review:

- `impressions_90d`
- `avg_position_90d`
- `days_since_last_update`

It does not use:

- `trend_direction`
- `trend_pct`
- future-window performance
- label-derived variables
- future content updates

Therefore, the baseline rule does not use future or label-derived information and provides a fair benchmark for later model work.

In [51]:
used_features = [
    "impressions_90d",
    "avg_position_90d",
    "days_since_last_update"
]

print("Features used:", used_features)
print("No future-window features used.")
print("No label-derived features used.")
print("No future content-update information used.")

Features used: ['impressions_90d', 'avg_position_90d', 'days_since_last_update']
No future-window features used.
No label-derived features used.
No future content-update information used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.